# TrustVLA RunPod LIBERO/OpenVLA Pilot

Pilot for selective-obedience evaluation: raw OpenVLA versus a trusted safety-policy gate. Use this notebook on a Linux GPU/SIM machine such as RunPod.

In [ ]:
from pathlib import Path
import subprocess
import os

PROJECT_DIR = Path.cwd()
SUITE = "libero_object"
LIMIT = 10
INIT_STATES = 3
HF_LOCAL_DIR = "/workspace/LIBERO-datasets"
MODEL_PATH = "openvla/openvla-7b"
DEVICE = "cuda:0"
MAX_STEPS = 50
RUN_ROLLOUTS = False  # Set True only after seed annotation is complete.

def run(cmd, check=True):
    print("$", " ".join(cmd))
    return subprocess.run(cmd, check=check, cwd=PROJECT_DIR)

print("Project:", PROJECT_DIR)

## 1. Environment Check

If `libero` is missing, install LIBERO before continuing to real rollout cells.

In [ ]:
run(["python", "-m", "trustvla.cli", "doctor"], check=False)

## 2. Selective Hugging Face Download

This downloads only one LIBERO suite, not the full dataset mirror.

In [ ]:
run([
    "python", "-m", "trustvla.cli", "download-libero-hf",
    "--suite", SUITE,
    "--local-dir", HF_LOCAL_DIR,
])

## 3. Export LIBERO Seed Draft

This requires LIBERO to be installed. The output needs manual annotation before benchmark generation.

In [ ]:
seed_path = f"data/{SUITE}_seed_draft.json"
run([
    "python", "-m", "trustvla.cli", "export-libero-seeds",
    "--suite", SUITE,
    "--limit", str(LIMIT),
    "--out", seed_path,
])
print("Annotate:", seed_path)

## 4. Generate Safety Policy And TrustVLA Pairs

Run this only after filling target objects, compatible distractors, absent objects, ambiguous targets, and hazards. Independently review the generated safety policy draft.

In [ ]:
benchmark_path = f"runs/{SUITE}/trustvla_pairs.jsonl"
safety_policy_path = f"data/{SUITE}_safety_policies_draft.json"
run([
    "python", "-m", "trustvla.cli", "export-safety-policies",
    "--seed-tasks", seed_path,
    "--out", safety_policy_path,
])
run([
    "python", "-m", "trustvla.cli", "generate",
    "--seed-tasks", seed_path,
    "--init-states", str(INIT_STATES),
    "--out", benchmark_path,
])
run([
    "python", "-m", "trustvla.cli", "validate-benchmark",
    "--benchmark", benchmark_path,
    "--safety-policies", safety_policy_path,
])
print("Benchmark:", benchmark_path)
print("Review safety policy:", safety_policy_path)

## 5. Raw And Safety-Gated OpenVLA Rollouts

Set `RUN_ROLLOUTS = True` in the config cell only after the benchmark file looks correct.

In [ ]:
if RUN_ROLLOUTS:
    run([
        "python", "-m", "trustvla.cli", "run-openvla-libero",
        "--benchmark", benchmark_path,
        "--out", f"runs/{SUITE}/openvla_rollouts.jsonl",
        "--model-path", MODEL_PATH,
        "--suite", SUITE,
        "--device", DEVICE,
        "--max-steps", str(MAX_STEPS),
        "--trace-dir", f"runs/{SUITE}/traces/openvla",
    ])
    run([
        "python", "-m", "trustvla.cli", "run-openvla-libero",
        "--benchmark", benchmark_path,
        "--out", f"runs/{SUITE}/openvla_language_emphasis.jsonl",
        "--model-path", MODEL_PATH,
        "--suite", SUITE,
        "--device", DEVICE,
        "--max-steps", str(MAX_STEPS),
        "--grounding-mode", "language_emphasis",
        "--trace-dir", f"runs/{SUITE}/traces/language_emphasis",
    ])
    run([
        "python", "-m", "trustvla.cli", "run-openvla-libero",
        "--benchmark", benchmark_path,
        "--out", f"runs/{SUITE}/openvla_gated_rollouts.jsonl",
        "--model-path", MODEL_PATH,
        "--suite", SUITE,
        "--device", DEVICE,
        "--max-steps", str(MAX_STEPS),
        "--grounding-mode", "language_emphasis",
        "--guarded",
        "--safety-policies", safety_policy_path,
        "--trace-dir", f"runs/{SUITE}/traces/openvla_gated",
    ])
else:
    print("Skipping rollout. Set RUN_ROLLOUTS=True after annotation is complete.")

## 6. Detect Events And Score Selective Obedience

Run after real rollout JSONL files exist.

In [ ]:
if RUN_ROLLOUTS:
    conditions = {
        "raw": f"runs/{SUITE}/openvla_rollouts.jsonl",
        "grounded": f"runs/{SUITE}/openvla_language_emphasis.jsonl",
        "guarded": f"runs/{SUITE}/openvla_gated_rollouts.jsonl",
    }
    detected = {}
    for label, rollout_path in conditions.items():
        detected[label] = rollout_path.replace(".jsonl", ".detected.jsonl")
        run([
            "python", "-m", "trustvla.cli", "detect-rollout-events",
            "--benchmark", benchmark_path,
            "--rollouts", rollout_path,
            "--out", detected[label],
        ])
        run([
            "python", "-m", "trustvla.cli", "tradeoff-score",
            "--benchmark", benchmark_path,
            "--rollouts", detected[label],
        ])
        run([
            "python", "-m", "trustvla.cli", "pair-score",
            "--benchmark", benchmark_path,
            "--rollouts", detected[label],
        ])
    run([
        "python", "-m", "trustvla.cli", "compare",
        "--benchmark", benchmark_path,
        "--rollout", f"raw={detected['raw']}",
        "--rollout", f"grounded={detected['grounded']}",
        "--rollout", f"guarded={detected['guarded']}",
        "--out", f"runs/{SUITE}/selective_obedience_report.md",
    ])
else:
    print("Skipping detection until real rollouts exist.")